In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList


class BanWordsLogitsProcessor(LogitsProcessor):

    def __init__(self, tokenizer, target_word="UNIX", max_occurrences=1, top_k=50):
        self.max_occurrences = max_occurrences
        self.top_k = top_k

        self.seq = tokenizer.encode(" " + target_word, add_special_tokens=False)
        self.seq_no_space = tokenizer.encode(target_word, add_special_tokens=False)

    def _count_full(self, ids):
        count = 0
        for seq in (self.seq, self.seq_no_space):
            n = len(seq)
            for i in range(len(ids) - n + 1):
                if ids[i:i + n] == seq:
                    count += 1
        return count

    def _prefix_len(self, ids):
        best = 0
        for seq in (self.seq, self.seq_no_space):
            for k in range(min(len(seq) - 1, len(ids)), 0, -1):
                if ids[-k:] == seq[:k]:
                    return max(best, k)
        return best

    def __call__(self, input_ids, scores):
        ids = input_ids[0].tolist()

        if self._count_full(ids) >= self.max_occurrences:
            return scores

        prefix_len = self._prefix_len(ids)

        if prefix_len == 0:
            firsts = {self.seq[0], self.seq_no_space[0]}
            top_ids = torch.topk(scores[0], self.top_k).indices.tolist()
            to_ban = [t for t in top_ids if t not in firsts]
            scores[:, to_ban] = -float("inf")
            return scores

        allowed = set()
        for seq in (self.seq, self.seq_no_space):
            if prefix_len < len(seq) and ids[-prefix_len:] == seq[:prefix_len]:
                allowed.add(seq[prefix_len])
        if allowed:
            mask = torch.full_like(scores, -float("inf"))
            mask[:, list(allowed)] = scores[:, list(allowed)]
            return mask

        return scores

In [10]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [19]:
import re
processor = BanWordsLogitsProcessor(tokenizer, "UNIX", max_occurrences=1, top_k=100)
logits_processor = LogitsProcessorList([processor])

inputs = tokenizer("The most popular server operating system is", return_tensors="pt")
outputs = model.generate(
    **inputs, max_new_tokens=60, do_sample=False,
    logits_processor=logits_processor, pad_token_id=tokenizer.eos_token_id,
)
text = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(text)
match_exact = re.search(r'\bUNIX\b', text) is not None

print("UNIX:", match_exact)

The most popular server operating system is supported Android platforms while both 'universal cloud over virtual operating with internet'; VMware drives each (UNIX, Windows, Linux, Mac OS X, etc.) with their own operating system.

The following table lists the supported operating systems and their respective versions.

Windows

Windows Server 2008
UNIX: True
